# Lab Sains Data P10 — Merapikan Data & SQL (Kelompok 2)
**Abrar Mohammad Fathur (12010223140022) · Muhammad Dezra Aria (12010223140038) · Samuel Pinondang Simanjuntak (12010223140072)**

Lanjutan P8–P9. Keputusan: label sentimen ulasan toko thrift diisi otomatis oleh AI (A), AI + cek manusia (B), atau manual (C).
Ambang: A bila akurasi ≥ 85% dan recall negatif ≥ 80%; B bila 70–85%; C bila < 70%.

Isi notebook:
1. Ambil data mentah (NusaX-senti) + label AI P9
2. Rapikan menjadi **tiga tabel bersih** (`ulasan`, `label_ai`, `label_manusia`); setiap langkah otomatis masuk **log pembersihan**
3. Tanyai tabel dengan **SQL di DuckDB** (cek integritas + 4 pertanyaan)

Jalankan dengan **Runtime → Run all**. Tidak perlu unggah berkas apa pun.

In [ ]:
# Sel 1 — pustaka dan konfigurasi
import io, re, sys, subprocess, urllib.request, datetime
import pandas as pd
try:
    import duckdb
except ImportError:                                   # Colab biasanya sudah punya duckdb
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
    import duckdb
print("pandas", pd.__version__, "| duckdb", duckdb.__version__)

TANGGAL_AMBIL = "2026-11-06"
URL_NUSAX = "https://raw.githubusercontent.com/IndoNLP/nusax/main/datasets/sentiment/indonesian/{}.csv"
URL_REPO  = "https://raw.githubusercontent.com/abrarmf/Lab-Sains-Data-B-Kelompok-2-/main/data/{}"
SPLIT_HARAP = {"train": 500, "valid": 100, "test": 400}
LABEL_SAH = {"negative", "neutral", "positive"}

def unduh_csv(url, cadangan):
    try:
        with urllib.request.urlopen(url, timeout=30) as r:
            return pd.read_csv(io.BytesIO(r.read()))
    except Exception as e:
        print(f"  gagal unduh {url.split('/')[-1]} ({e.__class__.__name__}) → pakai file lokal {cadangan}")
        return pd.read_csv(cadangan)

## 2. Log pembersihan
Fungsi `catat()` dipanggil di **setiap** langkah. Isinya: tabel, langkah, alasan, jumlah baris sebelum/sesudah, jumlah sel yang berubah. Log disimpan sebagai `log_pembersihan.csv`.

In [ ]:
LOG = []
def catat(tabel, langkah, alasan, sebelum, sesudah, diubah=0):
    LOG.append({"no": len(LOG) + 1, "tabel": tabel, "langkah": langkah, "alasan": alasan,
                "baris_sebelum": sebelum, "baris_sesudah": sesudah,
                "baris_dibuang": sebelum - sesudah, "sel_diubah": int(diubah)})
    print(f"[{len(LOG):02d}] {tabel:13s} {langkah:38s} {sebelum:>5} → {sesudah:<5} diubah={int(diubah)}")

## 3. Ambil data mentah

In [ ]:
bagian = []
for sp, n in SPLIT_HARAP.items():
    d = unduh_csv(URL_NUSAX.format(sp), f"{sp}.csv")
    assert len(d) == n, f"{sp}: {len(d)} baris, harusnya {n}"
    bagian.append(d.assign(split=sp))
raw = pd.concat(bagian, ignore_index=True)[["split", "id", "text", "label"]]
raw.to_csv(f"raw_{TANGGAL_AMBIL}.csv", index=False)          # file mentah tidak disentuh lagi
catat("ulasan", "gabung 3 split NusaX", "satu tabel 1 baris = 1 ulasan", 1000, len(raw))

ai_raw = unduh_csv(URL_REPO.format("label_ai_300.csv"), "label_ai_300.csv")
print("label AI P9:", ai_raw.shape)

## 4. Tabel `ulasan` — 1 baris = 1 ulasan
Langkah dikerjakan berurutan pada salinan; `raw` tidak diubah.

In [ ]:
u = raw.copy()

# L2 — kosong & id ganda: baris tanpa teks/label atau id ganda tidak bisa dipakai
n0 = len(u)
u = u.dropna(subset=["id", "text", "label"]).drop_duplicates(subset="id")
catat("ulasan", "buang baris kosong / id ganda", "baris tanpa teks/label tidak bisa dinilai", n0, len(u))

# L3 — rapikan spasi: spasi di awal/akhir dan spasi ganda membuat teks sama terlihat beda
rapi = u.text.str.strip().str.replace(r"\s+", " ", regex=True)
catat("ulasan", "rapikan spasi awal/akhir & ganda", "teks sama harus tertulis sama", len(u), len(u), (rapi != u.text).sum())
u["text"] = rapi

# L4 — duplikat setelah normalisasi (huruf kecil, tanpa tanda baca)
kunci = u.text.str.lower().str.replace(r"[^\w\s]", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
n0 = len(u)
u = u[~kunci.duplicated()]
catat("ulasan", "buang duplikat (huruf kecil, tanpa tanda baca)", "satu ulasan tidak boleh dihitung dua kali", n0, len(u))

# L5 — label harus salah satu dari 3 nilai sah
lab = u.label.str.strip().str.lower()
catat("ulasan", "seragamkan & validasi label", "label di luar 3 nilai merusak hitungan akurasi", len(u), len(u), (lab != u.label).sum())
u["label"] = lab
n0 = len(u); u = u[u.label.isin(LABEL_SAH)]
catat("ulasan", "buang label tidak sah", "hanya negative/neutral/positive", n0, len(u))

In [ ]:
# L6 — anonimisasi nama orang (lampu kuning → hijau). Daftar dari P8, + 2 nama tanda tangan CS yang ditemukan di P10
NAMA = ['dela','isna','diah','gina','tisna','echa','tian','luqman','elsa','adit','doni','rocky',
        'farrel','dian','nadia','lulung','rohayani','arif',
        'yogi','erf']                                   # tambahan P10
pola = re.compile(r'\b(' + '|'.join(NAMA) + r')\b', re.I)
ttd  = re.compile(r'([Tt]erima kasih,?\s+)(zan|era)(\s*\.?\s*)$')
u["n_nama_diganti"] = u.text.str.count(pola) + u.text.str.count(ttd)
baru = u.text.str.replace(pola, "[NAMA]", regex=True).str.replace(ttd, lambda m: m.group(1) + "[NAMA]" + m.group(3), regex=True)
catat("ulasan", "anonimisasi nama → [NAMA]", "nama orang tidak boleh masuk AI/publik", len(u), len(u), (baru != u.text).sum())
u["text"] = baru
sisa = u.text.str.contains(r'@\w+|https?://|www\.|(?:\+62|08)\d{7,}|\S+@\S+\.\w+', regex=True).sum()
assert sisa == 0, "masih ada pola kontak"

In [ ]:
# L7 — kolom turunan. domain = HEURISTIK kata kunci (bukan label manusia), dipakai untuk melihat
#      seberapa mirip data proksi dengan ulasan toko thrift
import re
DOMAIN = {  # urutan = prioritas saat jumlah kata kunci seri
 'pakaian_fashion': ['baju','kaos','celana','sepatu','tas','kemeja','jaket','pakaian','busana','jilbab','hijab','rok','dress','sandal','batik','fashion','distro','sweater','gaun'],
 'makanan_minuman': ['makan','makanan','restoran','resto','rasa','rasanya','enak','menu','masakan','porsi','minum','minuman','kopi','sate','bakso','nasi','ayam','mie','mi','sambal','kuliner','kafe','cafe','sup','soto','bebek','ikan','daging','kue','roti','pedas','gurih','lezat','teh','jus','es','steak','pizza','burger','seafood','warung','rumah makan','sarapan'],
 'telko_internet': ['sinyal','kuota','internet','provider','telkomsel','indosat','xl','pulsa','jaringan','wifi','indihome','operator','smartfren','tri','simpati','kartu perdana','paket data'],
 'transportasi': ['gojek','grab','ojek','ojol','driver','kereta','pesawat','bus','bis','taksi','penerbangan','maskapai','bandara','garuda','lion','travel','angkot','stasiun','tol','macet'],
 'bank_keuangan': ['bank','atm','rekening','transfer','kartu kredit','bca','mandiri','bni','bri','saldo','cicilan','kredit','tabungan','asuransi','pinjaman','teller','nasabah'],
 'hotel_wisata': ['hotel','kamar','wisata','pantai','pemandangan','liburan','kolam renang','penginapan','resort','villa','museum','taman','gunung','danau','pulau','wisatawan'],
 'belanja_toko': ['beli','membeli','barang','produk','toko','belanja','kirim','pengiriman','penjual','online','mall','diskon','harga','kasir','pesanan','order'],
 'gawai_elektronik': ['hp','handphone','ponsel','laptop','aplikasi','samsung','xiaomi','iphone','baterai','layar','kamera','tv','komputer','android'],
 'berita_politik': ['presiden','pemerintah','polisi','partai','dpr','menteri','gubernur','pilkada','pemilu','korban','tewas','kpk','korupsi','jokowi','prabowo','ahok','calon','warga','negara','undang'],
}
_pat = {k: re.compile(r'\b(' + '|'.join(map(re.escape, v)) + r')\b', re.I) for k, v in DOMAIN.items()}
def domain(teks):
    skor = {k: len(p.findall(teks)) for k, p in _pat.items()}
    best = max(skor.values())
    if best == 0: return 'lainnya'
    return next(k for k in DOMAIN if skor[k] == best)

u["n_kata"] = u.text.str.split().str.len()
u["panjang"] = pd.cut(u.n_kata, [0, 12, 30, 1000], labels=["pendek (≤12)", "sedang (13–30)", "panjang (>30)"]).astype(str)
u["domain"] = u.text.map(domain)
catat("ulasan", "tambah kolom n_kata, panjang, domain", "bahan pertanyaan SQL per kelompok", len(u), len(u))

# L8 — nama kolom & urutan akhir
ulasan = (u.rename(columns={"id": "id_ulasan", "text": "teks", "label": "label_pakar"})
           [["id_ulasan", "split", "teks", "label_pakar", "domain", "n_kata", "panjang", "n_nama_diganti"]]
           .sort_values("id_ulasan").reset_index(drop=True))
ulasan["id_ulasan"] = ulasan.id_ulasan.astype(int)
catat("ulasan", "ganti nama kolom & urutkan", "nama kolom jelas, 1 arti per kolom", len(ulasan), len(ulasan))
print(ulasan.domain.value_counts().to_dict())
ulasan.head()

## 5. Tabel `label_ai` — 1 baris = 1 label dari 1 model untuk 1 ulasan
Format panjang (*long*) supaya label dari model lain (mis. GLM/DeepSeek) cukup ditambah sebagai baris baru, bukan kolom baru.

In [ ]:
la = ai_raw.rename(columns={"id": "id_ulasan"})[["id_ulasan", "label_ai"]].copy()
n0 = len(la)
la["label_ai"] = la.label_ai.str.strip().str.lower()
la = la[la.label_ai.isin(LABEL_SAH)].drop_duplicates("id_ulasan")
la = la[la.id_ulasan.isin(ulasan.id_ulasan)]
catat("label_ai", "validasi label AI P9", "label sah, 1 label per ulasan, id ada di ulasan", n0, len(la))
la = la.assign(model="claude-opus-5.5", versi_prompt="P9-v1", tanggal_label="2026-11-06")
label_ai = la[["id_ulasan", "model", "versi_prompt", "tanggal_label", "label_ai"]].sort_values("id_ulasan").reset_index(drop=True)
catat("label_ai", "tambah kolom model, versi_prompt, tanggal", "setiap label bisa ditelusuri asalnya", len(label_ai), len(label_ai))
label_ai.head()

## 6. Tabel `label_manusia` — hasil cek 50 label (P9)
Diisi dari `cek50_terisi.csv` (lembar cek yang sudah diisi anggota). Selama belum ada, tabel kosong tetapi skemanya sudah siap.

In [ ]:
import os
kolom_lm = ["id_ulasan", "pemeriksa", "label_manusia"]
if os.path.exists("cek50_terisi.csv"):
    h = pd.read_csv("cek50_terisi.csv").rename(columns={"id": "id_ulasan", "id_nusax": "id_ulasan"})
    h["label_manusia"] = h.label_manusia.astype(str).str.strip().str.lower()
    h["pemeriksa"] = h.get("catatan / pemeriksa", h.get("catatan", "")).fillna("").astype(str)
    n0 = len(h); h = h[h.label_manusia.isin(LABEL_SAH)]
    label_manusia = h[kolom_lm].reset_index(drop=True)
    catat("label_manusia", "muat cek50_terisi.csv", "label kosong/tidak sah dibuang", n0, len(label_manusia))
else:
    label_manusia = pd.DataFrame(columns=kolom_lm).astype({"id_ulasan": int})
    print("cek50_terisi.csv belum ada → label_manusia kosong (0 baris)")

In [ ]:
# Simpan tabel bersih + log
ulasan.to_csv("ulasan_bersih.csv", index=False)
label_ai.to_csv("label_ai.csv", index=False)
label_manusia.to_csv("label_manusia.csv", index=False)
log = pd.DataFrame(LOG); log.to_csv("log_pembersihan.csv", index=False)
log

## 7. SQL di DuckDB
Tabel pandas didaftarkan ke DuckDB lalu ditanya dengan SQL biasa. Setiap query menjawab satu pertanyaan yang terkait keputusan.

In [ ]:
con = duckdb.connect()
for nama, df in {"ulasan": ulasan, "label_ai": label_ai, "label_manusia": label_manusia}.items():
    con.register(nama, df)
def sql(q):
    return con.sql(q).df()

**Q0 — Cek integritas.** Apakah tiap tabel bersih: tanpa id ganda, tanpa label tidak sah, dan setiap label AI punya pasangan ulasan?

In [ ]:
Q0 = """
SELECT 'ulasan'   AS tabel, COUNT(*) AS baris, COUNT(DISTINCT id_ulasan) AS id_unik,
       SUM(CASE WHEN label_pakar NOT IN ('negative','neutral','positive') THEN 1 ELSE 0 END) AS label_tidak_sah,
       0 AS tanpa_pasangan
FROM ulasan
UNION ALL
SELECT 'label_ai', COUNT(*), COUNT(DISTINCT a.id_ulasan),
       SUM(CASE WHEN a.label_ai NOT IN ('negative','neutral','positive') THEN 1 ELSE 0 END),
       SUM(CASE WHEN u.id_ulasan IS NULL THEN 1 ELSE 0 END)
FROM label_ai a LEFT JOIN ulasan u ON a.id_ulasan = u.id_ulasan
"""
sql(Q0)

**Q1 — Angka pembeda.** Seberapa sering AI benar, dan berapa persen keluhan (negatif) yang tertangkap?

In [ ]:
Q1 = """
SELECT COUNT(*)                                                         AS n,
       SUM(CASE WHEN a.label_ai = u.label_pakar THEN 1 ELSE 0 END)     AS benar,
       ROUND(100.0 * SUM(CASE WHEN a.label_ai = u.label_pakar THEN 1 ELSE 0 END) / COUNT(*), 1) AS akurasi_pct,
       ROUND(100.0 * SUM(CASE WHEN u.label_pakar = 'negative' AND a.label_ai = 'negative' THEN 1 ELSE 0 END)
                   / SUM(CASE WHEN u.label_pakar = 'negative' THEN 1 ELSE 0 END), 1)      AS recall_negatif_pct
FROM label_ai a JOIN ulasan u ON a.id_ulasan = u.id_ulasan
"""
q1 = sql(Q1); q1

**Q2 — Per domain.** Apakah AI sama bagusnya di semua topik? Berapa banyak ulasan pakaian di data proksi?

In [ ]:
Q2 = """
SELECT u.domain,
       COUNT(*)                                                                  AS n_dilabel_ai,
       ROUND(100.0 * SUM(CASE WHEN a.label_ai = u.label_pakar THEN 1 ELSE 0 END) / COUNT(*), 1) AS akurasi_pct,
       SUM(CASE WHEN u.label_pakar = 'negative' THEN 1 ELSE 0 END)              AS n_negatif
FROM label_ai a JOIN ulasan u ON a.id_ulasan = u.id_ulasan
GROUP BY u.domain
ORDER BY n_dilabel_ai DESC, u.domain
"""
sql(Q2)

**Q3 — Pola salah label.** Pasangan (label pakar → label AI) mana yang paling sering salah? Ini menentukan apa yang perlu dicek manusia di pilihan B.

In [ ]:
Q3 = """
SELECT u.label_pakar, a.label_ai, COUNT(*) AS jumlah,
       MIN(u.id_ulasan) AS contoh_id
FROM label_ai a JOIN ulasan u ON a.id_ulasan = u.id_ulasan
WHERE a.label_ai <> u.label_pakar
GROUP BY u.label_pakar, a.label_ai
ORDER BY jumlah DESC, u.label_pakar, a.label_ai
"""
q3 = sql(Q3); q3

**Q4 — Per panjang teks.** Ulasan toko thrift biasanya pendek. Apakah AI lebih sering salah di teks pendek?

In [ ]:
Q4 = """
SELECT u.panjang,
       COUNT(*) AS n,
       ROUND(100.0 * SUM(CASE WHEN a.label_ai = u.label_pakar THEN 1 ELSE 0 END) / COUNT(*), 1) AS akurasi_pct
FROM label_ai a JOIN ulasan u ON a.id_ulasan = u.id_ulasan
GROUP BY u.panjang
ORDER BY MIN(u.n_kata)
"""
sql(Q4)

**Q5 — Cek 50 manusia (aktif setelah `cek50_terisi.csv` ada).** Seberapa sering AI benar menurut anggota kelompok?

In [ ]:
Q5 = """
SELECT COUNT(*) AS n_dicek,
       SUM(CASE WHEN m.label_manusia = a.label_ai THEN 1 ELSE 0 END) AS ai_benar,
       ROUND(100.0 * SUM(CASE WHEN m.label_manusia = a.label_ai THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS akurasi_manusia_pct,
       ROUND(100.0 * SUM(CASE WHEN m.label_manusia = u.label_pakar THEN 1 ELSE 0 END) / NULLIF(COUNT(*), 0), 1) AS sepakat_pakar_pct
FROM label_manusia m
JOIN label_ai a ON m.id_ulasan = a.id_ulasan
JOIN ulasan  u ON m.id_ulasan = u.id_ulasan
"""
sql(Q5)

## 8. Terapkan ambang (sama dengan P8–P9)

In [ ]:
akurasi, rec_neg = q1.akurasi_pct[0] / 100, q1.recall_negatif_pct[0] / 100
if akurasi >= 0.85 and rec_neg >= 0.80:   pilihan = "A — label otomatis AI (dengan audit sampel berkala)"
elif akurasi >= 0.70:                      pilihan = "B — AI melabel, manusia cek label negatif/netral"
else:                                      pilihan = "C — tetap dibaca manual"
print(f"Akurasi {akurasi:.1%}, recall negatif {rec_neg:.1%} → {pilihan}")
n_fashion = int((ulasan.domain == "pakaian_fashion").sum())
print(f"Peringatan: hanya {n_fashion} dari {len(ulasan)} ulasan bertopik pakaian → keputusan tetap SEMENTARA sampai diuji di ulasan toko sendiri.")